<a href="https://colab.research.google.com/github/intelligent-environments-lab/CityLearn/blob/master/examples/load_environment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# How to Load the Current Project Environment

Este ejemplo carga el dataset versionado del proyecto actual: `citylearn_iquitos_2023_2025`. No descarga datasets externos ni usa nombres antiguos de CityLearn Challenge.


In [ ]:
# En este repositorio se recomienda usar el entorno local ya preparado.
# Si estas fuera del repo, instala CityLearn desde el codigo fuente local, no desde un dataset remoto:
# !python -m pip install -e ..


## Locate the Project Dataset

El dataset actual esta en `CityLearn/data/datasets/citylearn_iquitos_2023_2025/schema.json`. La siguiente celda funciona si el notebook se ejecuta desde `CityLearn/examples`, desde `CityLearn` o desde la raiz del superproyecto.


In [ ]:
# Configuracion portable para cargar el dataset actual del proyecto.
from pathlib import Path
import sys

PROJECT_DATASET_NAME = 'citylearn_iquitos_2023_2025'


def find_citylearn_root(start=None) -> Path:
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        direct_schema = candidate / 'data' / 'datasets' / PROJECT_DATASET_NAME / 'schema.json'
        nested_schema = candidate / 'CityLearn' / 'data' / 'datasets' / PROJECT_DATASET_NAME / 'schema.json'
        if direct_schema.is_file() and (candidate / 'citylearn').is_dir():
            return candidate
        if nested_schema.is_file() and (candidate / 'CityLearn' / 'citylearn').is_dir():
            return candidate / 'CityLearn'
    raise FileNotFoundError(
        f'No se encontro CityLearn/data/datasets/{PROJECT_DATASET_NAME}/schema.json desde {start}'
    )


CITYLEARN_ROOT = find_citylearn_root()
PROJECT_DATASET_ROOT = CITYLEARN_ROOT / 'data' / 'datasets' / PROJECT_DATASET_NAME
PROJECT_SCHEMA_PATH = PROJECT_DATASET_ROOT / 'schema.json'

if str(CITYLEARN_ROOT) not in sys.path:
    sys.path.insert(0, str(CITYLEARN_ROOT))

print('CITYLEARN_ROOT =', CITYLEARN_ROOT)
print('PROJECT_DATASET_NAME =', PROJECT_DATASET_NAME)
print('PROJECT_SCHEMA_PATH =', PROJECT_SCHEMA_PATH)
print('schema exists =', PROJECT_SCHEMA_PATH.is_file())

local_datasets = sorted(path.name for path in (CITYLEARN_ROOT / 'data' / 'datasets').iterdir() if path.is_dir())
print('Local datasets:', local_datasets)


Inicializa el entorno usando la ruta exacta de `schema.json`. Esta es la forma recomendada para este fork porque evita depender del registro interno `DataSet`.


In [ ]:
from citylearn.citylearn import CityLearnEnv

# Horizonte corto para prueba manual rapida; el entrenamiento completo usa 8760 pasos.
env = CityLearnEnv(str(PROJECT_SCHEMA_PATH), central_agent=True, episode_time_steps=24, random_seed=0, offline=True)
observations, info = env.reset(seed=0)

print('Buildings loaded:', len(env.buildings))
print('Observation groups:', len(observations))
print('Action space groups:', len(env.action_space))

env.close()


## Inspect the Local Dataset

El dataset ya esta versionado en el repositorio. Si necesitas inspeccionarlo o montarlo en Docker/AWS, copia la carpeta completa `CityLearn/data/datasets/citylearn_iquitos_2023_2025`.


In [ ]:
print('Dataset root:', PROJECT_DATASET_ROOT)
print('Files:')
for file_path in sorted(PROJECT_DATASET_ROOT.iterdir())[:25]:
    print('-', file_path.name)


## Load an Environment Using Schema Filepath

La ruta de schema tambien se puede pasar directamente a `CityLearnEnv`. Este bloque deja explicito el camino que deben usar los scripts manuales y contenedores Docker.


In [ ]:
from citylearn.citylearn import CityLearnEnv

schema_filepath = str(PROJECT_SCHEMA_PATH)
env = CityLearnEnv(schema_filepath, central_agent=True, episode_time_steps=24, random_seed=0, offline=True)
print('time_steps:', env.time_steps)
print('central_agent:', env.central_agent)
env.close()


Este enfoque tambien es el mas estable para una computadora nueva, porque solo depende de archivos locales del proyecto.


## Load an Environment Using Schema Dictionary Object

Tambien puedes cargar `schema.json` como `dict` para modificar parametros antes de construir el entorno. En ese caso fija `root_directory` al directorio real del dataset para que los CSV se resuelvan correctamente.


In [ ]:
from citylearn.citylearn import CityLearnEnv
from citylearn.utilities import FileHandler

schema = FileHandler.read_json(PROJECT_SCHEMA_PATH)
schema['root_directory'] = str(PROJECT_DATASET_ROOT)
env = CityLearnEnv(schema, central_agent=True, episode_time_steps=24, random_seed=0, offline=True)
print('Buildings loaded from dict:', len(env.buildings))
env.close()


Algunos parametros tambien se pueden pasar al constructor de `CityLearnEnv` para pruebas cortas sin alterar `schema.json`.


In [ ]:
from citylearn.citylearn import CityLearnEnv
from citylearn.utilities import FileHandler

schema = FileHandler.read_json(PROJECT_SCHEMA_PATH)
schema['root_directory'] = str(PROJECT_DATASET_ROOT)
env = CityLearnEnv(
    schema,
    central_agent=True,
    simulation_start_time_step=0,
    simulation_end_time_step=23,
    random_seed=0,
    offline=True,
)
print('Manual window time_steps:', env.time_steps)
env.close()
